# Gas Source

In [ ]:
import io
import urllib.request
from pathlib import Path

import openpyxl
import pandas as pd


Fetch daily gas commodity prices from AEMO's STTM and DWGM markets and forward-fill to 5-minute dispatch intervals.

- **NSW** → STTM Sydney hub ex ante price ($/GJ)
- **QLD** → STTM Brisbane hub ex ante price ($/GJ)
- **SA**  → STTM Adelaide hub ex ante price ($/GJ)
- **VIC** → DWGM Victoria beginning-of-day (6am schedule) price ($/GJ)


In [ ]:
"""
Datasource 4
 - Datasource origin: AEMO STTM & DWGM
 - Datasource name: gas prices
    STTM   : AEMO's Short Term Trading Market — daily ex ante commodity price ($/GJ) at each gas hub, set by the market clearing process for the upcoming gas day
    DWGM   : AEMO's Declared Wholesale Gas Market — 5-interval-per-day scheduled price ($/GJ) for Victoria; each interval covers a ~4-hour scheduling horizon (6am, 10am, 2pm, 6pm, 10pm)
 - Variables:
    gas_price_nsw : STTM Sydney hub ex ante daily commodity price ($/GJ)
    gas_price_qld : STTM Brisbane hub ex ante daily commodity price ($/GJ)
    gas_price_sa  : STTM Adelaide hub ex ante daily commodity price ($/GJ)
    gas_price_vic : DWGM Victoria scheduled price ($/GJ) — 5 intervals per gas day, forward-filled within each ~4-hour scheduling horizon
"""



def _fetch_gas_prices(start: str, end: str) -> pd.DataFrame:

    
    STTM_URL = "https://www.aemo.com.au/-/media/files/gas/sttm/data/sttm-price-and-withdrawals.xlsx"
    DWGM_URL = "https://www.aemo.com.au/-/media/files/gas/dwgm/dwgm-prices-and-demand.xlsx"

    STTM_HUBS = {
        "SYD price and withdrawals": "gas_price_nsw",
        "ADL price and withdrawals": "gas_price_sa",
        "BRI price and withdrawals": "gas_price_qld",
    }

    idx = pd.date_range(start=start, end=end, freq="5min", name="SETTLEMENTDATE")

    def _fetch_xlsx(url):
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=60) as resp:
            return io.BytesIO(resp.read())

    # --- STTM (NSW / QLD / SA) ---
    print("  Fetching STTM (SYD/ADL/BRI)...", flush=True)
    wb = openpyxl.load_workbook(_fetch_xlsx(STTM_URL), read_only=True, data_only=True)
    sttm_series = {}
    for sheet_name, col_name in STTM_HUBS.items():
        dates, prices = [], []
        for row in wb[sheet_name].iter_rows(min_row=2, values_only=True):  # row[0]=date, row[1]=exante_price
            if row[0] is not None and row[1] is not None:
                dates.append(pd.Timestamp(row[0]))
                prices.append(float(row[1]))
        # Daily — map every 5-min timestamp to its calendar date for lookup
        daily = pd.Series(prices, index=dates)
        expanded = daily.reindex(idx.normalize())
        expanded.index = idx
        sttm_series[col_name] = expanded.ffill().bfill()

    # --- DWGM (VIC) ---
    print("  Fetching DWGM (VIC)...", flush=True)
    wb = openpyxl.load_workbook(_fetch_xlsx(DWGM_URL), read_only=True, data_only=True)
    vic_index, vic_prices = [], []
    for row in wb["Prices"].iter_rows(min_row=2, values_only=True):  # row[0]=Gas_Date, row[1]=Hour, row[2]=Price
        if row[0] is not None and row[1] is not None and row[2] is not None:
            vic_index.append(pd.Timestamp(row[0]) + pd.Timedelta(hours=int(row[1])))
            vic_prices.append(float(row[2]))
    # Sub-daily — reindex directly; ffill carries each interval price forward ~4 hours
    dwgm = pd.Series(vic_prices, index=vic_index, name="gas_price_vic").sort_index()
    sttm_series["gas_price_vic"] = dwgm.reindex(idx, method="ffill").bfill()

    return pd.DataFrame(sttm_series, index=idx)


def main():
    processed_path = Path("Processed_data/4_gas_prices.csv")

    if processed_path.exists():
        print(f"  {processed_path.name} already exists — skipping.", flush=True)
        return pd.read_csv(processed_path, index_col=0, parse_dates=True).tail()

    df = _fetch_gas_prices(start="2018/01/01 00:05:00", end="2025/12/31 23:55:00")
    df.to_csv(processed_path)
    print(f"  Saved to {processed_path}.", flush=True)
    return df.tail()

main()


  Fetching STTM (SYD/ADL/BRI)...
  Fetching DWGM (VIC)...
  Expanding to 5-min intervals...
  Saved to Processed_data\4_gas_prices.csv.


,gas_price_nsw,gas_price_sa,gas_price_qld,gas_price_vic
SETTLEMENTDATE,,,,
2025-12-31 23:35:00,11.18,11.39,11.1,11.07
2025-12-31 23:40:00,11.18,11.39,11.1,11.07
2025-12-31 23:45:00,11.18,11.39,11.1,11.07
2025-12-31 23:50:00,11.18,11.39,11.1,11.07
2025-12-31 23:55:00,11.18,11.39,11.1,11.07
